# Real Data — NeurIPS 2023 Reproduction

Reproduces Table 2 and Figure 2 of the NeurIPS 2023 paper using `RidgeEM(t2=False)` (half-Cauchy prior on τ, matching the legacy `squareU=False` setting) and the `NEURIPS2023` named problem collections.

All experiment cells are tagged `skip-execution` and must be run manually.

## d = 1

In [ ]:
import numpy as np
from fastridge import RidgeEM, RidgeLOOCV
from experiments import run_real_data_experiments
from problems import NEURIPS2023
from data import DATASETS

estimators = {
    'EM':     RidgeEM(t2=False),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, base=10)),
}

problems_d1 = sorted(NEURIPS2023, key=lambda p: DATASETS[p.dataset]['n'])
results_d1 = run_real_data_experiments(
    problems_d1, estimators, n_iterations=100, seed=123, verbose=True)
print()

In [ ]:
import pandas as pd

rows_d1 = []
for problem, data_result in zip(problems_d1, results_d1):
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {'dataset': problem.dataset, 'target': problem.target, 'd': 1}
    row.update({est: data_result[est]['r2'] for est in data_result})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = data_result['EM']['p']
    row['n_train']        = data_result['EM']['n_train']
    row['n:p']            = data_result['EM']['n_train'] / data_result['EM']['p']
    rows_d1.append(row)
df_d1 = pd.DataFrame(rows_d1).sort_values('n_train').round(2)
df_d1

## d = 2

In [ ]:
from problems import NEURIPS2023_D2

problems_d2 = sorted(NEURIPS2023_D2, key=lambda p: DATASETS[p.dataset]['n'])
results_d2 = run_real_data_experiments(
    problems_d2, estimators, n_iterations=100, polynomial=2, seed=123, verbose=True)
print()

In [ ]:
rows_d2 = []
for problem, data_result in zip(problems_d2, results_d2):
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {'dataset': problem.dataset, 'target': problem.target, 'd': 2}
    row.update({est: data_result[est]['r2'] for est in data_result})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = data_result['EM']['p']
    row['n_train']        = data_result['EM']['n_train']
    row['n:p']            = data_result['EM']['n_train'] / data_result['EM']['p']
    rows_d2.append(row)
df_d2 = pd.DataFrame(rows_d2).sort_values('n_train').round(2)
df_d2

## d = 3

In [ ]:
from problems import NEURIPS2023_D3

problems_d3 = sorted(NEURIPS2023_D3, key=lambda p: DATASETS[p.dataset]['n'])
results_d3 = run_real_data_experiments(
    problems_d3, estimators, n_iterations=100, polynomial=3, seed=123, verbose=True)
print()

In [ ]:
rows_d3 = []
for problem, data_result in zip(problems_d3, results_d3):
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {'dataset': problem.dataset, 'target': problem.target, 'd': 3}
    row.update({est: data_result[est]['r2'] for est in data_result})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = data_result['EM']['p']
    row['n_train']        = data_result['EM']['n_train']
    row['n:p']            = data_result['EM']['n_train'] / data_result['EM']['p']
    rows_d3.append(row)
df_d3 = pd.DataFrame(rows_d3).sort_values('n_train').round(2)
df_d3

## Summary

In [ ]:
df_all = pd.concat([df_d1, df_d2, df_d3]).sort_values(['n_train', 'd']).round(2)
df_all

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def make_figure2(results_d1, results_d2, results_d3, output_path=None):
    """2x3 scatter of EM vs CV R² for d=1,2,3. Color = speed-up ratio.

    Top row: CV_glm on y-axis. Bottom row: CV_fix on y-axis.
    Values below CLIP_MIN=-0.1 are clipped to CLIP_MIN and shown with a dashed edge.
    Axis limits extend PAD beyond [CLIP_MIN, 1] so boundary points are fully visible.
    """
    results_list = [results_d1, results_d2, results_d3]
    cv_rows = [('CV_glm', 'CV GLM Grid'), ('CV_fix', 'CV Fixed Grid')]
    CLIP_MIN = -0.1
    PAD      =  0.03

    all_su = [
        dr[cv]['time'] / dr['EM']['time']
        for results in results_list
        for dr in results
        for cv, _ in cv_rows
    ]
    norm = mcolors.Normalize(vmin=min(all_su), vmax=max(all_su))
    cmap = plt.cm.Greens

    fig, axes = plt.subplots(2, 3, figsize=(9, 5.4), sharex=True, sharey=True)
    fig.subplots_adjust(left=0.11, right=0.84, bottom=0.11, top=0.93,
                        hspace=0.06, wspace=0.04)

    for col, results in enumerate(results_list):
        for row, (cv_name, cv_label) in enumerate(cv_rows):
            ax = axes[row, col]

            true_em = [dr['EM']['r2']                         for dr in results]
            true_cv = [dr[cv_name]['r2']                      for dr in results]
            su      = [dr[cv_name]['time'] / dr['EM']['time'] for dr in results]
            disp_em = [max(CLIP_MIN, x) for x in true_em]
            disp_cv = [max(CLIP_MIN, y) for y in true_cv]
            clipped = [e < CLIP_MIN or c < CLIP_MIN
                        for e, c in zip(true_em, true_cv)]
            colors  = cmap(norm(np.array(su)))

            idx_in  = [i for i, cl in enumerate(clipped) if not cl]
            idx_out = [i for i, cl in enumerate(clipped) if cl]

            if idx_in:
                ax.scatter([disp_em[i] for i in idx_in],
                            [disp_cv[i] for i in idx_in],
                            c=[colors[i] for i in idx_in],
                            s=50, zorder=3, edgecolors='k', linewidths=0.6)
            if idx_out:
                sc = ax.scatter([disp_em[i] for i in idx_out],
                                [disp_cv[i] for i in idx_out],
                                c=[colors[i] for i in idx_out],
                                s=50, zorder=4, edgecolors='k', linewidths=0.8)
                sc.set_linestyle('--')

            ax.plot([CLIP_MIN, 1], [CLIP_MIN, 1], 'k--', lw=0.8, zorder=2)
            ax.axhline(0, color='0.8', lw=0.5, zorder=1)
            ax.axvline(0, color='0.8', lw=0.5, zorder=1)
            ax.set_xlim(CLIP_MIN - PAD, 1 + PAD)
            ax.set_ylim(CLIP_MIN - PAD, 1 + PAD)

            if row == 1:
                ax.set_xlabel('BayesEM $R^2$')
                ax.set_xticks([0.0, 0.5, 1.0])
            if col == 0:
                ax.set_ylabel(f'{cv_label} $R^2$')
                ax.set_yticks([0.0, 0.5, 1.0])
            if row == 0:
                ax.set_title(f'$d = {col + 1}$')

    cbar_ax = fig.add_axes([0.86, 0.28, 0.02, 0.46])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.ax.yaxis.set_label_position('right')
    cbar.set_label('speed-up ratio', rotation=90, labelpad=-30)

    if output_path:
        fig.savefig(output_path, bbox_inches='tight')

make_figure2(results_d1, results_d2, results_d3,
             output_path='../output/paper2023_figure2.pdf')

In [ ]:
df_all.to_csv('../output/paper2023_table2.csv', index=False)